# Model Evaluation #

### 1. Calculate classification metrics ###

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load cleaned data
df = pd.read_csv("insurance_fraud_cleaned.csv")

# Separate features and target
X = df.drop("fraud reported", axis=1)
y = df["fraud reported"]

# Recreate the same train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Load the trained Random Forest model
model = joblib.load("random_forest_model2.pkl")

# Generate predictions
y_pred = model.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

Accuracy : 0.7790893760539629
Precision: 0.9264705882352942
Recall   : 0.10824742268041238
F1-score : 0.19384615384615383


### 2. Check for overfitting / underfitting ###

In [2]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("Training Accuracy:", train_score)
print("Testing Accuracy :", test_score)

if train_score > test_score + 0.10:
    print("\nResult: Possible Overfitting")
elif train_score < 0.60 and test_score < 0.60:
    print("\nResult: Possible Underfitting")
else:
    print("\nResult: Train and Test scores are relatively close")

Training Accuracy: 0.9998945926004006
Testing Accuracy : 0.7790893760539629

Result: Possible Overfitting


### 3. 5-Fold Cross-Validation ###

In [3]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross-Validation Scores:", cv_scores)
print("Average CV Score:", cv_scores.mean())
print("CV Score Spread:", cv_scores.max() - cv_scores.min())

Cross-Validation Scores: [0.78029505 0.77239199 0.77754349 0.77069056 0.77859779]
Average CV Score: 0.7759037757470756
CV Score Spread: 0.009604483369837435


### 4. Compare All Models ###

#### Logisitic regression ####

In [4]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
y_pred_lr = logistic_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print("Logistic Regression")
print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1-score :", lr_f1)

Logistic Regression
Accuracy : 0.5526981450252951
Precision: 0.2871111111111111
Recall   : 0.5549828178694158
F1-score : 0.37844171060339776


#### Decision Tree ####

In [6]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


In [7]:
y_pred_d = decision_tree_model.predict(X_test)

d_accuracy = accuracy_score(y_test, y_pred_d)
d_precision = precision_score(y_test, y_pred_d)
d_recall = recall_score(y_test, y_pred_d)
d_f1 = f1_score(y_test, y_pred_d)

print("Decision Tree")
print("Accuracy :", d_accuracy)
print("Precision:", d_precision)
print("Recall   :", d_recall)
print("F1-score :", d_f1)

Decision Tree
Accuracy : 0.6791736930860034
Precision: 0.3544715447154472
Recall   : 0.3745704467353952
F1-score : 0.3642439431913116


#### Comparision Table ####

In [8]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Accuracy": [
        accuracy,
        lr_accuracy,
        d_accuracy
    ],
    "Precision": [
        precision,
        lr_precision,
        d_precision
    ],
    "Recall": [
        recall,
        lr_recall,
        d_recall
    ],
    "F1-score": [
        f1,
        lr_f1,
        d_f1
    ]
})

print("\nModel Comparison:")
print(comparison)


Model Comparison:
                 Model  Accuracy  Precision    Recall  F1-score
0        Random Forest  0.779089   0.926471  0.108247  0.193846
1  Logistic Regression  0.552698   0.287111  0.554983  0.378442
2        Decision Tree  0.679174   0.354472  0.374570  0.364244


### 5. Hyperparameter Tuning ###

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print("GridSearchCV setup completed successfully!")

GridSearchCV setup completed successfully!


In [17]:
# Step 8: Run GridSearchCV

grid_search.fit(X_train, y_train)

print("GridSearchCV completed successfully!")
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1-score:", grid_search.best_score_)

GridSearchCV completed successfully!
Best Parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best CV F1-score: 0.37739652703565313


In [15]:
# Step 9: Create the tuned Random Forest model
from sklearn.ensemble import RandomForestClassifier
tuned_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42
)

tuned_model.fit(X_train, y_train)

print("Tuned Random Forest model trained successfully!")

Tuned Random Forest model trained successfully!


In [16]:
# Step 10: Evaluate the tuned Random Forest

y_pred_tuned = tuned_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned)
tuned_recall = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print("Tuned Random Forest")
print("Accuracy :", tuned_accuracy)
print("Precision:", tuned_precision)
print("Recall   :", tuned_recall)
print("F1-score :", tuned_f1)

Tuned Random Forest
Accuracy : 0.6091905564924115
Precision: 0.31746031746031744
Recall   : 0.5154639175257731
F1-score : 0.3929273084479371
